# 06 · Ajustar cuánto invertimos según el riesgo anterior

La dirección sigue viniendo de momentum o reversión. Ahora reducimos el tamaño cuando el riesgo previo es elevado, en lugar de aceptar o rechazar señales. Reunimos las alternativas sencillas antes del aprendizaje automático.

Necesitas Shiller, señales y componentes preparados como en los notebooks 01 y 02, con historia desde diciembre de 1924. No es necesario ejecutar antes los notebooks 03–05. La evaluación desde 1990 sigue deshabilitada.


## 1. Qué riesgo usamos y cómo cambia el tamaño

Se estima la desviación típica muestral de las 12 variaciones mensuales de precio anteriores, incluyendo el mes de decisión, y se anualiza multiplicando por raíz de 12. El dividendo reconstruido no se usa como predictor.

La escala del mes siguiente es `mínimo(exposición máxima, volatilidad objetivo / volatilidad estimada)`. Con objetivo 12 % y riesgo estimado 24 %, se aplica la mitad de la señal: +1 pasa a +0,5 y −1 a −0,5. Una señal cero sigue siendo efectivo. La exposición máxima inicial es 1, por lo que no hay apalancamiento. Riesgo exactamente cero da la exposición máxima; datos ausentes detienen el cálculo.

Es una adaptación a datos mensuales del índice. [Barroso y Santa-Clara (2015)](https://doi.org/10.1016/j.jfineco.2014.11.010) usan riesgo diario de una cartera transversal, aquí cambian cartera, frecuencia, ventana y límites. La regla principal de Moreira y Muir utiliza inversa de varianza, mientras que esta usa inversa de desviación típica. No son fórmulas intercambiables ni réplicas exactas. El objetivo del 12 % es una referencia de escala, no una promesa de riesgo realizado.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.config import DEFAULT_CONFIG, load_config
from tfm.data import load_monthly, read_result, compare_simple_strategies
from tfm.volatility import SIMPLE_STRATEGIES

CONFIG = DEFAULT_CONFIG
configuracion = load_config(CONFIG)
mensuales = load_monthly(configuracion)
componentes, _ = read_result(configuracion, "monthly_return_components")
senales, _ = read_result(configuracion, "shiller_signals")
parametros_riesgo = configuracion["volatility_scaling"]
display(pd.DataFrame([
    {"Supuesto": "Variaciones mensuales anteriores", "Valor": parametros_riesgo["lookback_months"]},
    {"Supuesto": "Objetivo anual (%)", "Valor": 100 * parametros_riesgo["target_annual_volatility"]},
    {"Supuesto": "Exposición máxima (%)", "Valor": 100 * parametros_riesgo["max_exposure"]},
]))
display(pd.DataFrame([configuracion["simulation"]]).T.rename(columns={0: "Supuesto de simulación"}))
print(f"Mismos meses para todas las cuentas: {componentes.period.iloc[0]} → {componentes.period.iloc[-1]}")


## 2. Comparar todas las alternativas

**Esta celda guarda o sustituye** `data/processed/simple_comparisons.parquet` y su JSON. Conserva las otras tablas y utiliza el mismo motor de cuentas. No descarga ni selecciona parámetros.

Son ocho cuentas: las tres básicas, las dos con veto bajista y las tres básicas ajustadas por volatilidad. Comprar y mantener ajustado ayuda a distinguir gestión del riesgo de utilidad de una señal. El veto bajista y el ajuste de tamaño son alternativas separadas y no se aplican juntos.

La volatilidad de la tabla es la observada en cada cuenta, que puede diferir del objetivo. La rotación cuenta entradas, reajustes y cierre. Los costes están en las unidades del capital normalizado. Los resultados no son operaciones ejecutables a los promedios de Shiller.


In [ ]:
resumen = compare_simple_strategies(CONFIG)
simulaciones, informe = read_result(configuracion, "simple_comparisons")
print(resumen["path"])
print(f"{informe['rows']} filas: ocho cuentas × {informe['months_per_strategy']} meses")
metricas = pd.DataFrame(informe["strategies"]).set_index("label")
vista = metricas[["annualized_return", "annualized_volatility", "max_drawdown", "sharpe", "annualized_turnover", "transaction_cost_total", "borrow_cost_total"]].copy()
for columna in ["annualized_return", "annualized_volatility", "max_drawdown"]:
    vista[columna] *= 100
vista = vista.rename(columns={"annualized_return": "Anualizada (%)", "annualized_volatility": "Volatilidad (%)",
                              "max_drawdown": "Caída máxima (%)", "sharpe": "Sharpe",
                              "annualized_turnover": "Rotación anual (veces)",
                              "transaction_cost_total": "Costes negociación", "borrow_cost_total": "Costes préstamo"})
display(vista.round(4))


## 3. Ver mayo de 1975

La decisión de abril usa las variaciones de mayo de 1974 a abril de 1975, es decir, necesita el precio de abril de 1974 como inicio de la ventana. No conoce el resultado de mayo de 1975.

La escala se aplica solo a las cuentas ajustadas. En las demás vale 1. El filtro bajista puede rechazar una señal, mientras que el ajuste de riesgo puede mantenerla con menor tamaño.

El largo parcial recibe dividendos por el importe comprado y RF por el efectivo restante. Un corto parcial paga dividendo y préstamo por el importe vendido y recibe RF sobre el capital aportado, sin interés adicional por los ingresos de la venta corta. Se cobran los cambios de tamaño aunque la dirección no cambie. Las cuentas ya tienen capitales distintos en 1975, así que compara también su rentabilidad porcentual.


In [ ]:
MES_OBJETIVO = "1975-05"
mes = simulaciones.loc[simulaciones.period.eq(MES_OBJETIVO)].set_index("strategy")
if len(mes) != len(SIMPLE_STRATEGIES):
    raise ValueError("Elige un mes dentro del desarrollo preparado.")
contexto = mes.loc["reversal_volatility"]
print(f"Decisión: {contexto.decision_period} | Mes objetivo: {MES_OBJETIVO}")
print(f"Precio inicial para calcular variaciones: {contexto.volatility_price_start_period}")
print(f"Riesgo anual estimado: {100 * contexto.estimated_annual_volatility:.3f} % | Escala: {contexto.volatility_scale:.4f}")
vista_mes = mes[["position", "volatility_scale", "net_return", "cash_interest", "borrow_cost", "transaction_cost"]].copy()
vista_mes["net_return"] *= 100
vista_mes = vista_mes.rename(index=SIMPLE_STRATEGIES, columns={"position": "Exposición", "volatility_scale": "Escala aplicada",
                                                             "net_return": "Neta del mes (%)", "cash_interest": "Remuneración RF",
                                                             "borrow_cost": "Préstamo", "transaction_cost": "Negociación"})
display(vista_mes.round(6))


## 4. Curvas de capital y caídas

Cada fila compara una regla sin ajustar, con filtro bajista y con escalado. Comprar y mantener se muestra como referencia. Las dos versiones de comprar y mantener también están en la tabla anterior. La escala logarítmica permite observar cuentas con evoluciones muy diferentes.


In [ ]:
ESCALA_LOGARITMICA = True
parametros = configuracion["simulation"]
evolucion = simulaciones.pivot(index="period", columns="strategy", values="capital_end")
mes_inicial = str(pd.Period(evolucion.index[0], freq="M") - 1)
inicial = pd.DataFrame({nombre: [parametros["initial_capital"]] for nombre in SIMPLE_STRATEGIES}, index=[mes_inicial])
evolucion = pd.concat([inicial, evolucion]).sort_index()
fechas = pd.PeriodIndex(evolucion.index, freq="M").to_timestamp(how="end")
caidas = 100 * (evolucion / evolucion.cummax() - 1)
fig, ejes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, layout="constrained")
for fila, regla in enumerate(["momentum", "reversal"]):
    for nombre, estilo in zip(["buy_and_hold", regla, f"{regla}_bear", f"{regla}_volatility"], [":", "--", "-.", "-"]):
        ejes[fila, 0].plot(fechas, evolucion[nombre], linestyle=estilo, label=SIMPLE_STRATEGIES[nombre])
        ejes[fila, 1].plot(fechas, caidas[nombre], linestyle=estilo, label=SIMPLE_STRATEGIES[nombre])
    if ESCALA_LOGARITMICA:
        ejes[fila, 0].set_yscale("log")
    ejes[fila, 0].set(title=SIMPLE_STRATEGIES[regla], ylabel="Capital normalizado")
    ejes[fila, 1].set(ylabel="Caída desde máximos (%)")
    for eje in ejes[fila]:
        eje.grid(alpha=0.25)
        eje.legend(fontsize=8)
for eje in ejes[-1]:
    eje.set_xlabel("Año")
fig.suptitle("Alternativas sencillas · desarrollo 1927–1989")
plt.show()


## 5. Qué sigue abierto

Una mejora puede proceder de invertir menos, pasar más tiempo en efectivo o cambiar costes y dividendos, sin que haya mejor predicción. El control de comprar y mantener ajustado ayuda a interpretar esa diferencia. No elegiremos después la ventana o el objetivo que más favorezcan el resultado.

Este bloque completa la primera versión de los comparadores sencillos. Todavía quedan variables, etiquetas, validación temporal, modelos y controles de sobreajuste. La sensibilidad de futuros modelos y el retraso de señales siguen pendientes. No se ha ejecutado la evaluación desde 1990 ni se ha demostrado robustez fuera de muestra.
